In [30]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [31]:
np.random.seed(0)
num_cases = 5000  
regions = ['APAC', 'EMEA', 'AMER']
client_types = ['Corporate', 'SME', 'Institutional']
stages = [
    '1_Document_Submission', 
    '2_KYC_AML_Check', 
    '3_Credit_Risk_Assessment', 
    '4_Legal_Compliance_Approval', 
    '5_Account_Activation'
]

In [32]:
data = []
start_date_base = datetime(2026, 1, 1)

In [33]:
for case_id in range(1001, 1001 + num_cases):
    region = random.choice(regions)
    client_type = random.choice(client_types)
    current_time = start_date_base + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))
    
    for stage in stages:
        if stage == '1_Document_Submission':
            proc_time = np.random.gamma(shape=2, scale=2)
        elif stage == '2_KYC_AML_Check':
            proc_time = np.random.gamma(shape=4, scale=4.5)
        elif stage == '3_Credit_Risk_Assessment':
            proc_time = np.random.gamma(shape=3, scale=3)
        elif stage == '4_Legal_Compliance_Approval':
            proc_time = np.random.gamma(shape=3.5, scale=4)
        else:
            proc_time = np.random.gamma(shape=1.5, scale=2)

        end_time = current_time + timedelta(hours=proc_time)
        error_prob = 0.22 if stage in ['2_KYC_AML_Check', '4_Legal_Compliance_Approval'] else 0.05
        error_flag = 1 if random.random() < error_prob else 0
        
        data.append({
            'Case_ID': f"CAS-{case_id}",
            'Client_Type': client_type,
            'Region': region,
            'Stage_Name': stage,
            'Start_Timestamp': current_time.strftime('%Y-%m-%d %H:%M:%S'),
            'End_Timestamp': end_time.strftime('%Y-%m-%d %H:%M:%S'),
            'Duration_Hours': round(proc_time, 2),
            'Error_Flag': error_flag
        })
        current_time = end_time

In [34]:
df = pd.DataFrame(data)
df.to_csv('corporate_onboarding_logs.csv', index=False)
print("Successfully generated 'corporate_onboarding_logs.csv' with 25,000 records!")

Successfully generated 'corporate_onboarding_logs.csv' with 25,000 records!


In [35]:
corporate_logs = pd.read_csv("corporate_onboarding_logs.csv")
corporate_logs

,Case_ID,Client_Type,Region,Stage_Name,Start_Timestamp,End_Timestamp,Duration_Hours,Error_Flag
0,CAS-1001,Corporate,AMER,1_Document_Submission,2026-06-27 06:00:00,2026-06-27 16:16:39,10.28,0
1,CAS-1001,Corporate,AMER,2_KYC_AML_Check,2026-06-27 16:16:39,2026-06-28 12:28:17,20.19,0
2,CAS-1001,Corporate,AMER,3_Credit_Risk_Assessment,2026-06-28 12:28:17,2026-06-29 09:33:05,21.08,0
3,CAS-1001,Corporate,AMER,4_Legal_Compliance_Approval,2026-06-29 09:33:05,2026-06-29 16:27:27,6.91,0
4,CAS-1001,Corporate,AMER,5_Account_Activation,2026-06-29 16:27:27,2026-06-29 18:34:30,2.12,0
...,...,...,...,...,...,...,...,...
24995,CAS-6000,SME,AMER,1_Document_Submission,2026-05-25 15:00:00,2026-05-25 20:53:36,5.89,0
24996,CAS-6000,SME,AMER,2_KYC_AML_Check,2026-05-25 20:53:36,2026-05-27 08:30:37,35.62,1
24997,CAS-6000,SME,AMER,3_Credit_Risk_Assessment,2026-05-27 08:30:37,2026-05-28 00:24:48,15.90,0
24998,CAS-6000,SME,AMER,4_Legal_Compliance_Approval,2026-05-28 00:24:48,2026-05-28 07:45:00,7.34,0


In [36]:
import sqlite3
import pandas as pd

df = pd.read_csv("corporate_onboarding_logs.csv")

conn = sqlite3.connect(":memory:")

df.to_sql("onboarding_logs", conn, index=False, if_exists="replace")

q1 = """
SELECT
    Stage_Name,
    COUNT(DISTINCT Case_ID) AS Total_Cases,
    ROUND(AVG(Duration_Hours), 2) AS Avg_Duration_Hours,
    SUM(Error_Flag) AS Total_Errors,
    ROUND(AVG(Error_Flag) * 100, 2) AS Error_Rate_Pct
FROM onboarding_logs
GROUP BY Stage_Name
ORDER BY Stage_Name;
"""
df_summary = pd.read_sql_query(q1, conn)
print("--- STAGE PERFORMANCE ---")
print(df_summary)

--- STAGE PERFORMANCE ---
                    Stage_Name  Total_Cases  Avg_Duration_Hours  Total_Errors  \
0        1_Document_Submission         5000                3.92           240   
1              2_KYC_AML_Check         5000               18.29          1052   
2     3_Credit_Risk_Assessment         5000                9.04           267   
3  4_Legal_Compliance_Approval         5000               13.96          1094   
4         5_Account_Activation         5000                2.99           280   

   Error_Rate_Pct  
0            4.80  
1           21.04  
2            5.34  
3           21.88  
4            5.60  


In [38]:
# --- PHASE 2: FINANCIAL & BUSINESS RISK CALCULATIONS ---

# 1. Business Rules & Assumptions

SLA_BY_CLIENT = {
    "SME": 36,
    "Corporate": 48,
    "Institutional": 72
}

PENALTY_BY_CLIENT = {
    "SME": 25,
    "Corporate": 50,
    "Institutional": 100
}

REWORK_COST_PER_ERROR = 100.0      # Manual correction cost per error

# 2. Calculate end-to-end onboarding duration for each client

case_summary = pd.read_sql_query("""
SELECT
    Case_ID,
    Client_Type,
    SUM(Duration_Hours) AS Total_Duration
FROM onboarding_logs
GROUP BY
    Case_ID,
    Client_Type
""", conn)

# Assign SLA and penalty based on client type
case_summary["SLA_Hours"] = case_summary["Client_Type"].map(SLA_BY_CLIENT)
case_summary["Penalty_Rate"] = case_summary["Client_Type"].map(PENALTY_BY_CLIENT)

# Validate mappings
if case_summary["SLA_Hours"].isnull().any():
    raise ValueError("One or more Client_Type values do not have an SLA mapping.")

if case_summary["Penalty_Rate"].isnull().any():
    raise ValueError("One or more Client_Type values do not have a penalty mapping.")

# 3. Calculate Operational Metrics

total_cases = df_summary["Total_Cases"].max()

total_avg_duration = case_summary["Total_Duration"].mean()

total_errors_stage_2_4 = df_summary[
    df_summary["Stage_Name"].str.contains("2_KYC|4_Legal")
]["Total_Errors"].sum()

# 4. Calculate SLA Performance

case_summary["Delay_Hours"] = (
    case_summary["Total_Duration"] -
    case_summary["SLA_Hours"]
).clip(lower=0)

total_sla_breach_hours = case_summary["Delay_Hours"].sum()

breached_cases = (case_summary["Delay_Hours"] > 0).sum()
within_sla_cases = total_cases - breached_cases

sla_compliance_rate = (
    within_sla_cases / total_cases
) * 100

avg_delay_hours = case_summary.loc[
    case_summary["Delay_Hours"] > 0,
    "Delay_Hours"
].mean()

max_delay_hours = case_summary["Delay_Hours"].max()

# 5. Estimate Business Impact

case_summary["Penalty"] = (
    case_summary["Delay_Hours"] *
    case_summary["Penalty_Rate"]
)

direct_financial_loss = case_summary["Penalty"].sum()

rework_cost = (
    total_errors_stage_2_4 *
    REWORK_COST_PER_ERROR
)

total_financial_impact = (
    direct_financial_loss +
    rework_cost
)

# 6. Display Business Summary

print("=== PHASE 2: FINANCIAL IMPACT SUMMARY ===")
print(f"Total Onboarding Cycle Time: {total_avg_duration:.2f} Hours")
print(f"Cases Within SLA:            {within_sla_cases}")
print(f"Cases Breaching SLA:         {breached_cases}")
print(f"SLA Compliance Rate:         {sla_compliance_rate:.2f}%")
print(f"Average Delay (Breached):    {avg_delay_hours:.2f} Hours")
print(f"Maximum Delay:               {max_delay_hours:.2f} Hours")
print(f"Total SLA Breach Hours:      {total_sla_breach_hours:,.0f} Hours")
print(f"Direct SLA Penalty Exposure: ${direct_financial_loss:,.2f}")
print(f"Rework Cost (Stage 2 & 4):   ${rework_cost:,.2f}")
print(f"Total Business Risk Impact:  ${total_financial_impact:,.2f}")

=== PHASE 2: FINANCIAL IMPACT SUMMARY ===
Total Onboarding Cycle Time: 48.20 Hours
Cases Within SLA:            2820
Cases Breaching SLA:         2180
SLA Compliance Rate:         56.40%
Average Delay (Breached):    14.25 Hours
Maximum Delay:               66.03 Hours
Total SLA Breach Hours:      31,075 Hours
Direct SLA Penalty Exposure: $1,028,303.50
Rework Cost (Stage 2 & 4):   $214,600.00
Total Business Risk Impact:  $1,242,903.50
